# 03 — Results analysis

Reads the completed runs and lays out the measured numbers in the order the dissertation argues them. No number here is typed by hand: everything displayed is read from a run directory, so each figure in the write-up traces back to `outputs/<run_id>/`.

The setup cell below declares exactly which runs the analysis reads. Re-run it after any new experiment so the sections downstream pick up the current runs.

Throughout, keep observations (what the measurements show) separate from explanations (why they may have occurred).

In [1]:
# Setup. This notebook is a PRESENTATION LAYER over src.evaluation.aggregation: run
# discovery, smoke-run exclusion, consolidation, and the recovery/degradation arithmetic
# all live in that module and are unit-tested in tests/test_aggregation_contracts.py.
# Every number displayed below is copied from a saved metrics file; nothing is recomputed
# under a different convention and nothing is entered by hand.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.evaluation.aggregation import (
    EXPERIMENT_METRIC_FILES,
    consolidate_runs,
    degradation_rows,
    discover_runs,
    load_metrics,
    reportable_runs,
    summarise_recovery,
    summarise_runs,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
OUTPUT_ROOT = REPO_ROOT / "outputs"

RECORDS = discover_runs(OUTPUT_ROOT)
REPORTABLE = reportable_runs(RECORDS)
CONSOLIDATED = consolidate_runs(REPORTABLE)


def newest(experiment_type):
    """Newest reportable run of one protocol, or None. Records are ordered oldest first."""
    matches = [record for record in REPORTABLE if record.experiment_type == experiment_type]
    return matches[-1] if matches else None


RUN_RECORDS = {name: newest(name) for name in EXPERIMENT_METRIC_FILES}
RUNS = {name: (record.run_dir if record else None) for name, record in RUN_RECORDS.items()}


def load(experiment_type):
    """(run_dir, metrics) for one protocol's newest reportable run, or (None, None)."""
    record = RUN_RECORDS[experiment_type]
    return (record.run_dir, load_metrics(record)) if record else (None, None)


def provenance(run):
    """Print the identity every reported number must be traceable to."""
    record = next((r for r in RECORDS if r.run_dir == run), None)
    if record is None:
        return
    print(f"run_id        : {record.run_id}")
    if record.is_synthetic_smoke:
        print("WARNING       : SYNTHETIC SMOKE RUN - pipeline evidence only, NOT results")
    packages = record.environment.get("packages", {})
    print(f"environment   : python {record.environment.get('python')} "
          f"torch {packages.get('torch')} transformers {packages.get('transformers')} "
          f"cuda={record.environment.get('cuda_available')}")
    print(f"config        : seed {record.seed}, {record.epochs} epochs, "
          f"lr {record.learning_rate}, batch {record.batch_size}, {record.model_name}")


# The full inventory, including runs that failed or were interrupted. A results chapter
# has to be able to say what did not finish, not only what did.
display(pd.DataFrame(summarise_runs(RECORDS, CONSOLIDATED))[
    ["run_id", "experiment_type", "status", "is_synthetic_smoke",
     "held_out_generator", "evaluated_conditions", "cells"]
])


,run_id,experiment_type,status,is_synthetic_smoke,held_out_generator,evaluated_conditions,cells
0,unseen_generator-20260808T144550443491Z-b386ea...,unseen_generator,completed,True,biggan,0,NaN
1,fine_tuning-20260808T144640138049Z-6ec6cf50a2-...,fine_tuning,completed,True,biggan,0,9.0
2,unseen_generator-20260808T150151025524Z-c74c3e...,unseen_generator,failed,False,biggan,0,NaN
3,unseen_generator-20260808T151948963300Z-c74c3e...,unseen_generator,completed,False,biggan,12,NaN
4,fine_tuning-20260808T171227711418Z-314210675e-...,fine_tuning,completed,False,biggan,25,5.0
5,baseline-20260809T150824861195Z-4c050175ac-2692,baseline,completed,False,NaN,9,NaN
6,ablation-20260809T194436211721Z-fc1a22d8e3-5389,ablation,incomplete,False,biggan,0,NaN


## 1. Baseline — the in-distribution reference

Performance with every generator present in training. This is the reference condition the later sections are measured against; on its own it says nothing about transfer to a new generator.

**What to check:** whether the training curves support the selected checkpoint rather than suggesting under- or overfitting; whether generator-wise values reveal a weakness the aggregate hides; and whether any dataset artefact could be inflating the number.

In [2]:
# 1. Baseline - in-distribution reference.
run, metrics = load("baseline")
if run is None:
    print("No completed baseline run yet.")
else:
    provenance(run)
    overall = metrics["overall"]
    display(pd.DataFrame([{
        "roc_auc": overall["roc_auc"], "average_precision": overall["average_precision"],
        "accuracy": overall["accuracy"], "precision": overall["precision"],
        "recall": overall["recall"], "f1": overall["f1"],
        "threshold": overall["threshold"], "support": overall["support"],
        "best_epoch": metrics.get("best_epoch"),
    }]).T.rename(columns={0: "value"}))
    display(pd.DataFrame(metrics["per_generator"]).T[
        ["support", "accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
    ].sort_values("f1"))

run_id        : baseline-20260809T150824861195Z-4c050175ac-2692
environment   : python 3.11.15 torch 2.13.0 transformers 5.14.1 cuda=False
config        : seed 42, 10 epochs, lr 0.001, batch 32, openai/clip-vit-base-patch32


,value
roc_auc,0.933925
average_precision,0.931407
accuracy,0.858000
precision,0.844042
recall,0.878286
f1,0.860823
threshold,0.500000
support,3500.000000
best_epoch,7.000000


,support,accuracy,precision,recall,f1,roc_auc,average_precision
real,1750.0,0.837714,0.000000,0.000,0.000000,NaN,NaN
vqdm,2000.0,0.819500,0.378556,0.692,0.489392,0.865241,0.468915
wukong,2000.0,0.834500,0.416838,0.812,0.550882,0.899417,0.593875
midjourney,2000.0,0.841000,0.432000,0.864,0.576000,0.931058,0.736571
stable_diffusion_v1_5,2000.0,0.841000,0.432000,0.864,0.576000,0.927205,0.635218
adm,2000.0,0.852000,0.455939,0.952,0.616580,0.963534,0.838467
glide,2000.0,0.855000,0.462121,0.976,0.627249,0.974917,0.866619
biggan,2000.0,0.856500,0.465160,0.988,0.632522,0.976103,0.824420


## 2. Unseen-generator performance — the generalisation gap

The same detector scored on a generator absent from training and from model selection, alongside the prevalence-matched in-distribution comparison. The difference between the two is the headline gap.

**What to check:** which error type moves most, and whether that conclusion depends on the threshold. Compare the threshold-free metrics against the threshold-dependent ones — ranking failure and calibration drift look different, and only the second is fixed by moving the cut-off.

In [3]:
# 2. Unseen-generator performance.
#
# Both test sets are balanced 50/50 over the same fixed real pool, so precision, F1 and
# PR-AUC are comparable between them. Threshold-free metrics are listed FIRST: a
# fixed-threshold gap on an unseen generator largely measures calibration drift.
run, metrics = load("unseen_generator")
if run is None:
    print("No completed unseen-generator run yet.")
else:
    provenance(run)
    print(f"held out      : {metrics['held_out_generator']}")
    print(f"known         : {', '.join(metrics['known_generators'])}")
    print(f"seed          : {metrics['seed']}")
    print(f"manifest      : {metrics['manifest_sha256'][:32]}...")
    gap = metrics["generalisation_gap"]
    rows = []
    for label, key in (("roc_auc (threshold-free)", "roc_auc"),
                       ("average_precision (threshold-free)", "average_precision"),
                       ("f1 @ default threshold", "f1_at_default_threshold")):
        entry = gap.get(key, {})
        rows.append({"metric": label, "in_distribution": entry.get("in_distribution"),
                     "unseen": entry.get("unseen"), "absolute_drop": entry.get("absolute_drop")})
    selected = gap.get("f1_at_baseline_validation_selected_threshold", {})
    if "unseen" in selected:
        rows.append({"metric": "f1 @ validation-selected threshold",
                     "in_distribution": None, "unseen": selected["unseen"],
                     "absolute_drop": None})
    display(pd.DataFrame(rows))

    print("\ntest composition and thresholds (provenance):")
    display(pd.DataFrame([metrics["final_test_composition"]]).T.rename(columns={0: "value"}))
    display(pd.DataFrame(metrics["thresholds"]).T)

run_id        : unseen_generator-20260808T151948963300Z-c74c3e0db3-363b
environment   : python 3.11.15 torch 2.13.0 transformers 5.14.1 cuda=False
config        : seed 42, 10 epochs, lr 0.001, batch 32, openai/clip-vit-base-patch32
held out      : biggan
known         : vqdm, stable_diffusion_v1_5, wukong, adm, glide, midjourney
seed          : 42
manifest      : 3a5d4aa23f77cebc514c0362da3bfe6a...


,metric,in_distribution,unseen,absolute_drop
0,roc_auc (threshold-free),0.937376,0.928416,0.008960
1,average_precision (threshold-free),0.941380,0.924260,0.017121
2,f1 @ default threshold,0.864542,0.846154,0.018388
3,f1 @ validation-selected threshold,NaN,0.834615,NaN



test composition and thresholds (provenance):


,value
final_test_sha256,6791012c43e9f645729b5f99db38c4c44d66b76b3ed3ca...
held_out_fake_count,250
policy,balanced_50_50_fixed_real_pool
positive_prevalence,0.5
real_count,250
real_pool_available,1750
real_pool_sha256,3e80924f63f5035baf86967e7f1d699a213ebc31fa2ecd...
real_test_pool_seed,20260808


,held_out_samples_used,provenance,selection_metric,selection_sample_count,selection_score,value
baseline_validation_selected,0,selected_on_seen_generator_validation_only__gr...,f1,3250,0.834744,0.37
default,NaN,fixed_prior_from_config_model.decision_threshold,NaN,NaN,NaN,0.5


## 3. Fine-tuning recovery — what limited labelled data buys

Performance at each budget relative to the measured 0% condition, with the actual labelled sample counts those percentages represent.

**What to check:** whether improvement is monotonic within uncertainty; whether it appears in both metric families or only one; and whether it plateaus, stays incomplete, or trades precision against recall. Report the spread across declared seeds rather than a selected best run.

In [4]:
# 3. Fine-tuning recovery.
#
# labelled_images_consumed counts EVERY labelled held-out image the cell used, including
# the adaptation-validation images spent on model selection and threshold selection.
run, metrics = load("fine_tuning")
if run is None:
    print("No completed fine-tuning run yet.")
else:
    provenance(run)
    print(f"held out          : {metrics['held_out_generator']}")
    print(f"adaptation pool   : {metrics['adaptation_pool_size']}")
    print(f"final test        : {metrics['final_unseen_test_size']} "
          f"(prevalence {metrics['final_test_composition']['positive_prevalence']})")
    print(f"baseline threshold: {metrics['baseline_threshold']} "
          f"({metrics['baseline_threshold_provenance']})")
    print(f"starting ckpt     : {metrics['starting_checkpoint_compatibility'].get('starting_checkpoint_metadata', {})}")

    cells = pd.read_csv(run / "recovery_cells.csv")
    columns = [c for c in [
        "adaptation_percentage", "subset_seed", "training_seed",
        "adaptation_train_count", "adaptation_validation_count", "labelled_images_consumed",
        "roc_auc", "average_precision",
        "f1", "f1_at_adaptation_threshold", "f1_at_baseline_threshold",
        "threshold_adaptation_selected", "threshold_baseline_unchanged",
    ] if c in cells.columns]
    display(cells[columns].sort_values("adaptation_percentage"))

    print("\nacross-seed summary (from the run's own summary block):")
    summary = pd.DataFrame(metrics["summary"]["rows"])
    if not summary.empty:
        display(summary[summary["metric"].isin(["f1", "roc_auc"])])

    # Interpretation guard: if ROC-AUC is already flat while F1 moves, the curve is
    # measuring calibration, not adaptation. State which one the text is claiming.
    if "roc_auc" in cells.columns and cells["roc_auc"].notna().any():
        spread = cells["roc_auc"].max() - cells["roc_auc"].min()
        print(f"\nROC-AUC spread across all budgets: {spread:.4f}")
        if spread < 0.01:
            print("  ROC-AUC is essentially flat -> any F1 movement is a THRESHOLD effect,")
            print("  not evidence that adaptation improved the ranking.")

run_id        : fine_tuning-20260808T171227711418Z-314210675e-90fa
environment   : python 3.11.15 torch 2.13.0 transformers 5.14.1 cuda=False
config        : seed 42, 10 epochs, lr 0.001, batch 32, openai/clip-vit-base-patch32
held out          : biggan
adaptation pool   : 15999
final test        : 500 (prevalence 0.5)
baseline threshold: 0.37 (selected_on_seen_generator_validation_only__grid_search__no_held_out_samples)
starting ckpt     : {'epoch': 9, 'fine_tune_mode': 'head_only', 'model_name': 'openai/clip-vit-base-patch32', 'seed': 42, 'validation_metric_name': 'f1', 'validation_metric_value': 0.8278606965174129}


,adaptation_percentage,subset_seed,training_seed,adaptation_train_count,adaptation_validation_count,labelled_images_consumed,roc_auc,average_precision,f1,f1_at_adaptation_threshold,f1_at_baseline_threshold,threshold_adaptation_selected,threshold_baseline_unchanged
0,0.00,NaN,NaN,0,0,0,0.928416,0.924260,0.846154,0.834615,0.834615,NaN,0.37
1,0.05,42.0,42.0,640,160,800,0.984288,0.983275,0.851936,0.855204,0.898048,0.45,0.37
2,0.10,42.0,42.0,1280,320,1600,0.993792,0.993456,0.895197,0.941909,0.936975,0.32,0.37
3,0.20,42.0,42.0,2560,640,3200,0.996048,0.995818,0.930233,0.937238,0.957055,0.47,0.37
4,0.50,42.0,42.0,6400,1600,8000,0.997504,0.997456,0.963265,0.961145,0.979920,0.53,0.37



across-seed summary (from the run's own summary block):


,adaptation_percentage,fine_tune_mode,labelled_images_consumed,maximum,mean,metric,minimum,recovery_vs_zero_percent,runs,standard_deviation
0,0.05,head_only,[800],0.851936,0.851936,f1,0.851936,0.005782,1,0.0
2,0.05,head_only,[800],0.984288,0.984288,roc_auc,0.984288,NaN,1,0.0
4,0.10,head_only,[1600],0.895197,0.895197,f1,0.895197,0.049043,1,0.0
6,0.10,head_only,[1600],0.993792,0.993792,roc_auc,0.993792,NaN,1,0.0
8,0.20,head_only,[3200],0.930233,0.930233,f1,0.930233,0.084079,1,0.0
10,0.20,head_only,[3200],0.996048,0.996048,roc_auc,0.996048,NaN,1,0.0
12,0.50,head_only,[8000],0.963265,0.963265,f1,0.963265,0.117111,1,0.0
14,0.50,head_only,[8000],0.997504,0.997504,roc_auc,0.997504,NaN,1,0.0
16,0.00,none,[0],0.846154,0.846154,f1,0.846154,0.000000,1,0.0
18,0.00,none,[0],0.928416,0.928416,roc_auc,0.928416,NaN,1,0.0



ROC-AUC spread across all budgets: 0.0691


## 4. Fine-tuning-depth ablation — how much of CLIP must change

The same budgets repeated at three adaptation depths: head-only, last block, and full. Head-only success would indicate the frozen CLIP features already separate the new generator; gains from deeper modes would indicate representation change is needed.

**What to check:** that starting weights, subset IDs, final-test IDs, training budget, and selection rule were held constant. Under-performance by the full mode is at least as likely to reflect small-data overfitting or an unsuitable learning rate as a lack of capacity.

In [5]:
# 4. Fine-tuning-depth ablation.
run, metrics = load("ablation")
if run is None:
    print("No completed ablation run yet (run it on ONE representative held-out generator).")
else:
    provenance(run)
    controls = metrics["controls"]
    print(f"modes            : {', '.join(metrics['modes'])}")
    print(f"budget policy    : {controls['training_budget_policy']}")
    print(f"epoch-budget violations: {controls['budget_policy_violated_by'] or 'none'}")
    print(f"mode overrides   : {controls['mode_overrides_applied'] or 'none'}")
    print(f"shared final test: {controls['final_test_sample_id_count']} samples")
    display(pd.DataFrame(metrics["mode_comparison"]))
    print("\ntrainable parameters per mode:")
    display(pd.DataFrame([
        {"mode": mode, "trainable_tensors": len(names)}
        for mode, names in metrics["trainable_parameter_names_by_mode"].items()
    ]))
    print("\n" + metrics["interpretation_note"])

No completed ablation run yet (run it on ONE representative held-out generator).


## 5. Cross-experiment observations

Pulls the experiments together into one table: the observed pattern, the runs and figures supporting it, the competing explanations, and the further test that would discriminate between them.

Candidate confounds worth separating here: generator heterogeneity, content and domain shift, dataset artefacts, class prevalence, calibration, subset variance, and compute trade-offs. Avoid causal language unless the experimental control genuinely supports it.

In [6]:
# 5. Cross-experiment observations, computed by src.evaluation.aggregation so the same
# arithmetic backs the notebook, scripts/build_report.py, and the tests.
#
# Three recovery quantities are reported because none is safe alone:
#   absolute_recovery     metric(p) - metric(0%)
#   relative_improvement  that difference as a fraction of the 0% value
#   gap_closed_fraction   that difference as a fraction of the measured
#                         in-distribution-minus-unseen gap -- flagged unreliable when the
#                         gap is too small for the ratio to carry information.
if not CONSOLIDATED:
    print("No reportable runs to draw observations from yet.")
else:
    degradation = pd.DataFrame(degradation_rows(CONSOLIDATED))
    if not degradation.empty:
        print("Generalisation gap (in-distribution vs held-out generator):")
        display(
            degradation[degradation["operating_point"] == "default"][
                ["held_out_generator", "metric", "in_distribution", "unseen",
                 "absolute_drop", "relative_drop", "run_id"]
            ]
        )

    recovery = pd.DataFrame(summarise_recovery(CONSOLIDATED, records=REPORTABLE))
    if recovery.empty:
        print("\nNo adaptation cells yet.")
    else:
        print("\nRecovery against labelled budget (threshold-free metric first):")
        display(
            recovery[
                (recovery["metric"] == "roc_auc")
                & (recovery["operating_point"] == "default")
            ][
                ["held_out_generator", "fine_tune_mode", "adaptation_percentage", "runs",
                 "mean", "standard_deviation", "standard_error", "zero_percent_reference",
                 "absolute_recovery", "relative_improvement", "in_distribution_reference",
                 "gap_closed_fraction", "gap_closed_is_reliable"]
            ]
        )

        # Interpretation guard. If the threshold-free ranking metric is already flat while
        # F1 moves, the curve is measuring calibration, not adaptation. State which one
        # the write-up is claiming.
        auc = recovery[(recovery["metric"] == "roc_auc")
                       & (recovery["operating_point"] == "default")]["mean"]
        if not auc.empty:
            spread = float(auc.max() - auc.min())
            print(f"\nROC-AUC spread across adapted budgets: {spread:.4f}")
            if spread < 0.01:
                print("  Essentially flat -> any F1 movement is a THRESHOLD effect, not")
                print("  evidence that adaptation improved the ranking.")

        unreliable = recovery[recovery["gap_closed_is_reliable"] == False]  # noqa: E712
        if not unreliable.empty:
            print(f"\n{len(unreliable)} summary rows have a gap too small for "
                  "'fraction of the gap closed' to be meaningful; quote absolute_recovery "
                  "for those instead.")


Generalisation gap (in-distribution vs held-out generator):


,held_out_generator,metric,in_distribution,unseen,absolute_drop,relative_drop,run_id
0,biggan,roc_auc,0.937376,0.928416,0.008960,0.009559,unseen_generator-20260808T151948963300Z-c74c3e...
1,biggan,average_precision,0.941380,0.924260,0.017121,0.018187,unseen_generator-20260808T151948963300Z-c74c3e...
2,biggan,f1,0.864542,0.846154,0.018388,0.021269,unseen_generator-20260808T151948963300Z-c74c3e...
3,biggan,accuracy,0.864000,0.848000,0.016000,0.018519,unseen_generator-20260808T151948963300Z-c74c3e...
4,biggan,precision,0.861111,0.856557,0.004554,0.005288,unseen_generator-20260808T151948963300Z-c74c3e...
5,biggan,recall,0.868000,0.836000,0.032000,0.036866,unseen_generator-20260808T151948963300Z-c74c3e...



Recovery against labelled budget (threshold-free metric first):


,held_out_generator,fine_tune_mode,adaptation_percentage,runs,mean,standard_deviation,standard_error,zero_percent_reference,absolute_recovery,relative_improvement,in_distribution_reference,gap_closed_fraction,gap_closed_is_reliable
12,biggan,head_only,0.05,1,0.984288,0.0,None,0.928416,0.055872,0.060180,0.937376,6.235714,False
30,biggan,head_only,0.10,1,0.993792,0.0,None,0.928416,0.065376,0.070417,0.937376,7.296429,False
48,biggan,head_only,0.20,1,0.996048,0.0,None,0.928416,0.067632,0.072847,0.937376,7.548214,False
66,biggan,head_only,0.50,1,0.997504,0.0,None,0.928416,0.069088,0.074415,0.937376,7.710714,False



ROC-AUC spread across adapted budgets: 0.0132

28 summary rows have a gap too small for 'fraction of the gap closed' to be meaningful; quote absolute_recovery for those instead.


## 7. What is missing

A results chapter must state what was *not* measured, so a gap is never read as a result.
This cell derives that list from the runs actually present, and it is the same list
`python -m scripts.build_report` writes to
`outputs/report/consolidated/missing_results.md`.


In [7]:
# Coverage of the experimental design, derived from what is on disk.
incomplete = [r for r in RECORDS if r.status != "completed"]
if incomplete:
    print("Runs that did not complete:")
    for record in incomplete:
        print(f"  {record.run_id}  [{record.status}]")
else:
    print("Every discovered run completed.")

for protocol in EXPERIMENT_METRIC_FILES:
    if RUN_RECORDS[protocol] is None:
        print(f"MISSING PROTOCOL: no reportable {protocol!r} run")

held_out = sorted({r.held_out_generator for r in REPORTABLE
                   if r.experiment_type == "unseen_generator" and r.held_out_generator})
baseline_generators = sorted({
    row["generator"] for row in CONSOLIDATED
    if row["experiment_type"] == "baseline" and row["generator"]
    and row["generator"] != "real"
})
if baseline_generators:
    remaining = [g for g in sorted(set(baseline_generators) | set(held_out)) if g not in held_out]
    print(f"\nHeld out so far      : {held_out or 'none'}")
    print(f"Not yet held out     : {remaining or 'none'}")

modes = sorted({row["fine_tune_mode"] for row in CONSOLIDATED
                if row["experiment_type"] in {"fine_tuning", "ablation"}
                and row["fine_tune_mode"] and row["fine_tune_mode"] != "none"})
print(f"Fine-tune depths run : {modes or 'none'}")
print(f"Depths still missing : "
      f"{[m for m in ('head_only', 'last_block', 'full') if m not in modes] or 'none'}")

seeds = sorted({(row["subset_seed"], row["training_seed"]) for row in CONSOLIDATED
                if row["subset_seed"] is not None})
print(f"\n(subset_seed, training_seed) pairs: {seeds or 'none'}")
if len(seeds) <= 1:
    print("  Only one seed pair -> across-seed variability is UNMEASURED. Standard")
    print("  deviations of 0.0 in the tables above mean 'not measured', not 'no spread',")
    print("  and no error bars are drawn anywhere in this notebook.")


Runs that did not complete:
  unseen_generator-20260808T150151025524Z-c74c3e0db3-21d6  [failed]
  ablation-20260809T194436211721Z-fc1a22d8e3-5389  [incomplete]
MISSING PROTOCOL: no reportable 'ablation' run

Held out so far      : ['biggan']
Not yet held out     : ['adm', 'glide', 'midjourney', 'stable_diffusion_v1_5', 'vqdm', 'wukong']
Fine-tune depths run : ['head_only']
Depths still missing : ['last_block', 'full']

(subset_seed, training_seed) pairs: [(42, 42)]
  Only one seed pair -> across-seed variability is UNMEASURED. Standard
  deviations of 0.0 in the tables above mean 'not measured', not 'no spread',
  and no error bars are drawn anywhere in this notebook.


## 6. Discussion, validity, and limitations

**Internal validity** — leakage audits, checkpoint and threshold selection, partition isolation, repeated seeds, and whether each comparison held its intended controls constant.

**Construct validity** — what "generalisation" and "recovery" mean operationally here. A single-dataset comparison may combine generator, content, provenance, and compression shifts into one measured difference.

**External validity** — claims are limited to the generators studied, the real-image domain, the post-processing conditions, the model family, and the data budgets tested. New generator versions and deployment prevalence may differ.

**Statistical conclusion validity** — sample sizes, the number of seeds actually run, and the uncertainty attached to every reported difference.